# RQ06 — Comparação das métricas por fator (#171)

**Quais características dos repositórios estão associadas a um melhor desempenho DORA?**

Comparamos três fatores definidos na hipótese do artigo: **estrelas**, **número de contribuidores** e **idade do repositório em dias**, agrupados em quartis. São seis colunas de métricas: frequência de deploy, lead time (a)/(b), CFR (a)/(b) e tempo de recuperação, totalizando **18 testes globais planejados**.

A unidade é o repositório. O CSV final é a única entrada de dados. Nesta #171 são calculados testes e **p-valores brutos**. A correção de Holm, ε² e Cliff's delta pertencem à **#172**; as decisões nominais aqui não são conclusões confirmadas após correção.

## Reprodução e parâmetros

Instale as dependências de `requirements.txt` e execute todas as células a partir de `Lab03/` ou `Lab03/analise/`. Não há consultas à API nem uso de tokens. Para outro CSV do mesmo contrato, defina `LAB03_RQ06_CSV` antes de iniciar o kernel. `LAB03_RQ06_SAIDA` configura uma pasta isolada de saída. Caminhos relativos usam a pasta de trabalho do kernel.

O limiar **nominal** é α=0,05. Kruskal-Wallis usa aproximação qui-quadrado com correção de empates; Mann-Whitney é bilateral, assintótico, com correção de empates e continuidade. Uma comparação requer pelo menos cinco valores válidos em **cada** grupo; caso contrário, fica não estimável com motivo. No CSV publicado os grupos são maiores que esse mínimo.

Referências oficiais: [Kruskal-Wallis](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kruskal.html), [Mann-Whitney](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html) e [quartis com pandas.qcut](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html).

In [1]:
import hashlib
import json
import os
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from IPython.display import Markdown, display

candidatas = [p / sufixo for p in (Path.cwd(), *Path.cwd().parents) for sufixo in ("", "Lab03")]
RAIZ = next(p for p in candidatas if (p / "pipeline/comparacao_rq06.py").is_file())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from pipeline.comparacao_rq06 import ALPHA, FATORES, METRICAS, MINIMO_GRUPO, analisar, carregar_dataset

DATASET = Path(os.environ.get("LAB03_RQ06_CSV", str(RAIZ / "entregas/s02-03-dataset-500-final/dataset-500.csv"))).expanduser().resolve()
SAIDA = Path(os.environ.get("LAB03_RQ06_SAIDA", str(RAIZ / "analise"))).expanduser().resolve()
TABELAS = SAIDA / "tabelas"
ARQUIVOS = {chave: f"rq06_{chave}.csv" for chave in ("grupos", "cortes", "resumos", "globais", "pares")}
destinos = {TABELAS / nome for nome in (*ARQUIVOS.values(), "rq06_proveniencia.json")}
if DATASET in {p.resolve() for p in destinos}:
    raise ValueError("Saída não pode sobrescrever o CSV de entrada.")
TABELAS.mkdir(parents=True, exist_ok=True)
dados = carregar_dataset(DATASET)
sha_entrada = dados.attrs["csv_sha256"]
pd.options.display.float_format = "{:.6g}".format
print(f"Repositórios: {len(dados)} | α nominal: {ALPHA} | SHA-256: {sha_entrada}")
display(dados[list(FATORES)].describe())

Repositórios: 500 | α nominal: 0.05 | SHA-256: 2552ba6b0779e895d01bea88742cb668ee1812a8369a224ed5281708140b30bd


,estrelas,contribuidores,idade_dias
count,500,500,500
mean,59855.9,761.982,2530.83
std,39760,1109.21,1754.48
min,30088,1,12
25%,36047.8,180.75,939
50%,45049,395,2390.5
75%,68388.5,887.25,3954.5
max,391470,9944,6746


## Definição dos quartis

Os cortes são calculados **uma vez na população com valor do fator**, antes de excluir nulos das métricas. Portanto, lead time ou CFR ausente não muda os grupos dos outros testes. Os limites são quantis 0%, 25%, 50%, 75% e 100% de `pandas.qcut`, com interpolação linear. Os intervalos são fechados à direita; o primeiro também inclui o menor valor. O registro dos limites usa os números completos, sem a aproximação textual dos intervalos do pandas.

Empates permanecem no mesmo grupo, sem desempate por ID. Limites repetidos são removidos (`duplicates="drop"`); quatro grupos não são forçados artificialmente. Se houver menos de quatro grupos ocupados, eles recebem rótulos G1…Gk e o motivo é registrado. Com dois grupos válidos aplica-se Mann-Whitney diretamente; com menos de dois, o teste não é estimável. Fator constante ou totalmente ausente também é identificado.

No CSV final de 500 repositórios, os três fatores formam Q1–Q4 com 125 projetos por grupo. Essa composição é recalculada caso o CSV seja substituído.

In [2]:
resultados = analisar(dados, alpha=ALPHA)
for fator in FATORES:
    display(Markdown(f"### {fator}"))
    display(resultados["cortes"].loc[resultados["cortes"]["fator"] == fator])
display(resultados["grupos"].head(10))

### estrelas

,fator,n_populacao,n_fator,ausentes_fator,grupo,limite_inferior,limite_superior,inclui_inferior,n_grupo,grupos_efetivos,motivo
0,estrelas,500,500,0,Q1,30088,36047.8,True,125,4,None
1,estrelas,500,500,0,Q2,36047.8,45049,False,125,4,None
2,estrelas,500,500,0,Q3,45049,68388.5,False,125,4,None
3,estrelas,500,500,0,Q4,68388.5,391470,False,125,4,None


### contribuidores

,fator,n_populacao,n_fator,ausentes_fator,grupo,limite_inferior,limite_superior,inclui_inferior,n_grupo,grupos_efetivos,motivo
4,contribuidores,500,500,0,Q1,1,180.75,True,125,4,None
5,contribuidores,500,500,0,Q2,180.75,395,False,125,4,None
6,contribuidores,500,500,0,Q3,395,887.25,False,125,4,None
7,contribuidores,500,500,0,Q4,887.25,9944,False,125,4,None


### idade_dias

,fator,n_populacao,n_fator,ausentes_fator,grupo,limite_inferior,limite_superior,inclui_inferior,n_grupo,grupos_efetivos,motivo
8,idade_dias,500,500,0,Q1,12,939,True,125,4,None
9,idade_dias,500,500,0,Q2,939,2390.5,False,125,4,None
10,idade_dias,500,500,0,Q3,2390.5,3954.5,False,125,4,None
11,idade_dias,500,500,0,Q4,3954.5,6746,False,125,4,None


,id,full_name,estrelas,grupo_estrelas,contribuidores,grupo_contribuidores,idade_dias,grupo_idade_dias
0,8514,rails/rails,58838,Q3,7088,Q4,6746,Q4
1,156018,redis/redis,76813,Q4,961,Q4,6402,Q4
2,237159,expressjs/express,69569,Q4,394,Q2,6305,Q4
3,458058,symfony/symfony,31191,Q1,4949,Q4,6113,Q4
4,507775,elastic/elasticsearch,78265,Q4,2560,Q4,6078,Q4
5,529502,scrapy/scrapy,64808,Q3,750,Q3,6064,Q4
6,557980,socketio/socket.io,63220,Q3,574,Q3,6047,Q4
7,569041,curl/curl,43177,Q2,1593,Q4,6040,Q4
8,571770,jgm/pandoc,46598,Q3,662,Q3,6038,Q4
9,576201,mrdoob/three.js,116151,Q4,2576,Q4,6035,Q4


## Cobertura e estatísticas por grupo

Cada combinação fator × métrica retém somente projetos com ambos os valores. Não se imputam zeros, não se removem outliers e não se exige que outras métricas estejam preenchidas. Zeros observados são preservados. Os indicadores globais de coleta incompleta não eliminam projetos automaticamente, pois não identificam a métrica afetada; a análise descreve as métricas disponíveis no CSV.

A tabela registra n, ausentes, mediana e quartis por grupo. As métricas temporais estão em **horas**, DF em **releases/semana**, CFR em **fração**. A idade do fator usa **dias**. Contribuidores e estrelas correspondem ao momento da coleta; não são medidas históricas de toda a janela.

In [3]:
globais = resultados["globais"]
display(globais[["fator", "metrica", "n_populacao", "ausentes_fator", "ausentes_metrica_com_fator", "n", "grupos", "menor_grupo"]])
for fator in FATORES:
    display(Markdown(f"### Distribuições por quartil: {fator}"))
    display(resultados["resumos"].loc[resultados["resumos"]["fator"] == fator,
            ["metrica", "grupo", "n_grupo", "n", "ausentes_metrica_no_grupo", "q1", "mediana", "q3"]])

,fator,metrica,n_populacao,ausentes_fator,ausentes_metrica_com_fator,n,grupos,menor_grupo
0,estrelas,deployment_frequency,500,0,0,500,4,125
1,estrelas,lead_time_a,500,0,1,499,4,124
2,estrelas,lead_time_b,500,0,1,499,4,124
3,estrelas,cfr_a,500,0,0,500,4,125
4,estrelas,cfr_b,500,0,2,498,4,123
5,estrelas,tempo_recuperacao,500,0,31,469,4,115
6,contribuidores,deployment_frequency,500,0,0,500,4,125
7,contribuidores,lead_time_a,500,0,1,499,4,124
8,contribuidores,lead_time_b,500,0,1,499,4,124
9,contribuidores,cfr_a,500,0,0,500,4,125


### Distribuições por quartil: estrelas

,metrica,grupo,n_grupo,n,ausentes_metrica_no_grupo,q1,mediana,q3
0,deployment_frequency,Q1,125,125,0,0.210959,0.40274,0.939726
1,deployment_frequency,Q2,125,125,0,0.230137,0.383562,0.939726
2,deployment_frequency,Q3,125,125,0,0.249315,0.479452,0.958904
3,deployment_frequency,Q4,125,125,0,0.268493,0.728767,1.41918
4,lead_time_a,Q1,125,125,0,81.7694,271.887,865.61
5,lead_time_a,Q2,125,125,0,100.184,263.306,854.304
6,lead_time_a,Q3,125,124,1,122.841,340.752,964.845
7,lead_time_a,Q4,125,125,0,65.5975,216.573,997.983
8,lead_time_b,Q1,125,125,0,72.3092,363.187,1145.34
9,lead_time_b,Q2,125,125,0,70.7085,267.085,1175.27


### Distribuições por quartil: contribuidores

,metrica,grupo,n_grupo,n,ausentes_metrica_no_grupo,q1,mediana,q3
24,deployment_frequency,Q1,125,125,0,0.210959,0.460274,0.843836
25,deployment_frequency,Q2,125,125,0,0.230137,0.441096,1.28493
26,deployment_frequency,Q3,125,125,0,0.210959,0.460274,1.05479
27,deployment_frequency,Q4,125,125,0,0.268493,0.517808,1.24658
28,lead_time_a,Q1,125,125,0,27.44,127.29,437.816
29,lead_time_a,Q2,125,125,0,63.5489,235.513,643.741
30,lead_time_a,Q3,125,125,0,122.706,278.898,941.494
31,lead_time_a,Q4,125,124,1,266.55,793.363,4079.78
32,lead_time_b,Q1,125,125,0,32.4103,97.5653,307.371
33,lead_time_b,Q2,125,125,0,58.3217,201.615,612.152


### Distribuições por quartil: idade_dias

,metrica,grupo,n_grupo,n,ausentes_metrica_no_grupo,q1,mediana,q3
48,deployment_frequency,Q1,125,125,0,0.364384,0.843836,2.05205
49,deployment_frequency,Q2,125,125,0,0.249315,0.594521,1.4
50,deployment_frequency,Q3,125,125,0,0.191781,0.326027,0.728767
51,deployment_frequency,Q4,125,125,0,0.210959,0.345205,0.747945
52,lead_time_a,Q1,125,125,0,15.0811,72.8228,196.083
53,lead_time_a,Q2,125,125,0,75.4506,216.801,652.131
54,lead_time_a,Q3,125,124,1,169.616,450.213,1057.89
55,lead_time_a,Q4,125,125,0,262.139,998.297,4050.58
56,lead_time_b,Q1,125,125,0,25.5239,70.8863,147.989
57,lead_time_b,Q2,125,125,0,82.8919,294.841,651.183


## Testes globais por fator e métrica

Com três ou mais grupos válidos, calculamos H e p-valor de Kruskal-Wallis. Rejeitar o teste global não identifica os pares responsáveis pela diferença. Os testes investigam diferenças nas distribuições; interpretar uma diferença como mudança de mediana requer atenção às formas e dispersões dos grupos.

Os testes não estimáveis continuam entre os 18 planejados, com motivo, sem receber p=0. Quando todos os valores são iguais entre três ou mais grupos, Kruskal-Wallis é indefinido pela correção de empates: o caso é registrado, sem executar pós-testes. Ausência de diferença nominal não comprova equivalência.

In [4]:
display(globais[["fator", "metrica", "teste", "n", "estatistica", "p_valor_bruto", "alpha", "diferenca_nominal", "motivo"]])
nominais = globais.loc[globais["poshoc_executado"]]
print(f"Testes globais planejados: {len(globais)}; estimáveis: {globais['p_valor_bruto'].notna().sum()}; "
      f"Kruskal-Wallis que disparam pós-testes nominais: {len(nominais)}.")

,fator,metrica,teste,n,estatistica,p_valor_bruto,alpha,diferenca_nominal,motivo
0,estrelas,deployment_frequency,kruskal_wallis,500,8.13425,0.0433171,0.05,True,None
1,estrelas,lead_time_a,kruskal_wallis,499,1.71584,0.633417,0.05,False,None
2,estrelas,lead_time_b,kruskal_wallis,499,1.16333,0.761812,0.05,False,None
3,estrelas,cfr_a,kruskal_wallis,500,3.69743,0.296044,0.05,False,None
4,estrelas,cfr_b,kruskal_wallis,498,5.13504,0.162172,0.05,False,None
5,estrelas,tempo_recuperacao,kruskal_wallis,469,4.32817,0.228139,0.05,False,None
6,contribuidores,deployment_frequency,kruskal_wallis,500,3.1869,0.363697,0.05,False,None
7,contribuidores,lead_time_a,kruskal_wallis,499,74.2056,5.36232e-16,0.05,True,None
8,contribuidores,lead_time_b,kruskal_wallis,499,108.712,2.07715e-23,0.05,True,None
9,contribuidores,cfr_a,kruskal_wallis,500,4.47961,0.214116,0.05,False,None


Testes globais planejados: 18; estimáveis: 18; Kruskal-Wallis que disparam pós-testes nominais: 8.


## Mann-Whitney par a par

Quando Kruskal-Wallis retorna **p bruto < 0,05**, calculamos todos os pares não ordenados dos grupos daquela combinação (seis pares com Q1–Q4). Não selecionamos somente pares favoráveis. A tabela registra o **U da primeira amostra**, grupos, n de cada lado e p-valor bilateral. Se os grupos têm a mesma forma, uma diferença pode ser interpretada como localização; o teste não se limita à igualdade de medianas em geral.

Com exatamente dois grupos, o U e seu p aparecem na tabela global como `mann_whitney_direto`; não se repete a mesma comparação como pós-teste. A razão de executar ou não os pares consta em `motivo_poshoc`, inclusive quando falta evidência.

**Integração com #172:** usar estes p-valores brutos para aplicar Holm na família declarada e calcular ε²/Cliff's delta. A seleção nominal dos pós-testes não elimina a necessidade dessa correção nem autoriza interpretar estes achados como diferenças confirmadas. O critério final de seleção e relato dos pares deverá considerar a correção, preservando estes resultados originais.

In [5]:
display(globais[["fator", "metrica", "p_valor_bruto", "poshoc_executado", "motivo_poshoc"]])
display(resultados["pares"])
print(f"Comparações par a par realizadas: {len(resultados['pares'])}.")

,fator,metrica,p_valor_bruto,poshoc_executado,motivo_poshoc
0,estrelas,deployment_frequency,0.0433171,True,p_bruto_menor_alpha
1,estrelas,lead_time_a,0.633417,False,sem_diferenca_nominal
2,estrelas,lead_time_b,0.761812,False,sem_diferenca_nominal
3,estrelas,cfr_a,0.296044,False,sem_diferenca_nominal
4,estrelas,cfr_b,0.162172,False,sem_diferenca_nominal
5,estrelas,tempo_recuperacao,0.228139,False,sem_diferenca_nominal
6,contribuidores,deployment_frequency,0.363697,False,sem_diferenca_nominal
7,contribuidores,lead_time_a,5.36232e-16,True,p_bruto_menor_alpha
8,contribuidores,lead_time_b,2.07715e-23,True,p_bruto_menor_alpha
9,contribuidores,cfr_a,0.214116,False,sem_diferenca_nominal


,fator,metrica,grupo_1,grupo_2,n_1,n_2,U_1,p_valor_bruto,diferenca_nominal,alpha,metodo
0,estrelas,deployment_frequency,Q1,Q2,125,125,7586.5,0.693147,False,0.05,assintotico_bilateral_empates_continuidade
1,estrelas,deployment_frequency,Q1,Q3,125,125,7175,0.264987,False,0.05,assintotico_bilateral_empates_continuidade
2,estrelas,deployment_frequency,Q1,Q4,125,125,6355.5,0.0108207,True,0.05,assintotico_bilateral_empates_continuidade
3,estrelas,deployment_frequency,Q2,Q3,125,125,7341.5,0.41035,False,0.05,assintotico_bilateral_empates_continuidade
4,estrelas,deployment_frequency,Q2,Q4,125,125,6515,0.0232521,True,0.05,assintotico_bilateral_empates_continuidade
5,estrelas,deployment_frequency,Q3,Q4,125,125,6905,0.112535,False,0.05,assintotico_bilateral_empates_continuidade
6,contribuidores,lead_time_a,Q1,Q2,125,125,6252,0.00635693,True,0.05,assintotico_bilateral_empates_continuidade
7,contribuidores,lead_time_a,Q1,Q3,125,125,5195,4.70113e-06,True,0.05,assintotico_bilateral_empates_continuidade
8,contribuidores,lead_time_a,Q1,Q4,125,124,3254,2.55278e-15,True,0.05,assintotico_bilateral_empates_continuidade
9,contribuidores,lead_time_a,Q2,Q3,125,125,6796,0.0755346,False,0.05,assintotico_bilateral_empates_continuidade


Comparações par a par realizadas: 48.


## Leitura e limites

A lista abaixo é gerada com os testes nominais desta execução. Não substitui a correção de Holm e os tamanhos de efeito da #172. Um p-valor pequeno não informa a magnitude ou a importância prática da diferença e não estabelece causalidade. Fatores podem estar associados entre si; estes testes isolados não ajustam confusão entre idade, tamanho e popularidade.

CFR (b) ainda depende da validação humana completa da heurística (#162/#163). Falhas de CI em CFR (a) não são necessariamente incidentes em produção. O tempo de recuperação agregado só existe quando há recuperação observável. As ausências e a seleção de projetos podem afetar a interpretação.

In [6]:
linhas = [f"- {r.fator} × {r.metrica}: H={r.estatistica:.6g}, p bruto={r.p_valor_bruto:.6g}, n={int(r.n)}."
          for r in nominais.itertuples(index=False)]
display(Markdown("\n".join(linhas) if linhas else "Nenhum Kruskal-Wallis disparou pós-testes nominais nesta execução."))

- estrelas × deployment_frequency: H=8.13425, p bruto=0.0433171, n=500.
- contribuidores × lead_time_a: H=74.2056, p bruto=5.36232e-16, n=499.
- contribuidores × lead_time_b: H=108.712, p bruto=2.07715e-23, n=499.
- idade_dias × deployment_frequency: H=44.1678, p bruto=1.39024e-09, n=500.
- idade_dias × lead_time_a: H=140.775, p bruto=2.57268e-30, n=499.
- idade_dias × lead_time_b: H=147.189, p bruto=1.06428e-31, n=499.
- idade_dias × cfr_b: H=84.9685, p bruto=2.63578e-18, n=498.
- idade_dias × tempo_recuperacao: H=18.2762, p bruto=0.000385758, n=469.

## Exportação e proveniência

As cinco tabelas preservam a população e os limites dos grupos, resumos, testes globais e pares. São entradas para a #172 sem depender de copiar valores do notebook. O JSON registra hash do CSV e do código, janela, parâmetros, bibliotecas e contagens. A entrada é conferida antes de exportar e não é alterada.

In [7]:
if hashlib.sha256(DATASET.read_bytes()).hexdigest() != sha_entrada:
    raise RuntimeError("CSV mudou durante a análise; reexecute desde o início.")
for chave, nome in ARQUIVOS.items():
    resultados[chave].to_csv(TABELAS / nome, index=False, float_format="%.17g", lineterminator="\n")
proveniencia = {
    "issue": 171,
    "csv": DATASET.relative_to(RAIZ).as_posix() if DATASET.is_relative_to(RAIZ) else str(DATASET),
    "csv_sha256": sha_entrada,
    "total_repositorios": len(dados),
    "janela": {"inicio": dados['janela_inicio'].iloc[0], "fim_exclusivo": dados['janela_fim_exclusivo'].iloc[0]},
    "fatores": list(FATORES), "metricas": list(METRICAS), "alpha_nominal": ALPHA,
    "agrupamento": "qcut(q=4, duplicates=drop); cortes fixados antes de excluir nulos das métricas; empates não separados por ID",
    "minimo_por_grupo": MINIMO_GRUPO,
    "teste_global": "Kruskal-Wallis assintótico com empates; Mann-Whitney direto se houver dois grupos",
    "posthoc": "todos os pares não ordenados quando p bruto de KW < alpha; Mann-Whitney bilateral assintótico com empates e continuidade",
    "multiplicidade": "p-valores brutos; Holm e tamanhos de efeito na issue #172; resultados nominais",
    "contagens": {"testes_planejados": len(globais), "testes_estimaveis": int(globais['p_valor_bruto'].notna().sum()),
                  "globais_com_poshoc": int(globais['poshoc_executado'].sum()), "pares_realizados": len(resultados['pares'])},
    "codigo_sha256": hashlib.sha256((RAIZ / 'pipeline/comparacao_rq06.py').read_bytes()).hexdigest(),
    "bibliotecas": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__},
}
(TABELAS / "rq06_proveniencia.json").write_text(json.dumps(proveniencia, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Exportadas cinco tabelas e proveniência; entrada preservada.")

Exportadas cinco tabelas e proveniência; entrada preservada.
